<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/LangChaing(ToolCalls).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.5/124.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 22.5 MB/s eta 0:00:00


In [23]:
from google.colab import userdata
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import SecretStr
from typing import List

def print_conversation(conversation: List[AIMessage]):
    for message in conversation:
        message.pretty_print()

In [48]:
@tool
def do_nothing() -> str:
    """
    This tool does absolutely nothing. Do not call it!
    """
    return "Nothing..."

@tool
def get_interesting_fact() -> str:
    """
    This tool will discover an interesting fact to you.
    """

    return "The Earth is actually not a perfect sphere."

@tool
def get_database_status(cluster_name: str):
    """
    Returns information about the current database status.

    Args:
      cluster_name: The cluster name of \"default\" if unknown.
    """
    return "healthy"

In [65]:
tools = [do_nothing, get_interesting_fact, get_database_status]
tools_registry = { t.name: t for t in tools }

# Tool_Choice = auto is default one - the model choosese
# Tool_Choice = any - the model chooses but at least 1 tool call
# Tool_Choice = forced - подаваш името на инструмента като задължителен за изпълнение, извикване
# forced - се прави в изолирани случаи tool_choice=do_nothing.name
openai_api_key = SecretStr(userdata.get("OPENAI_API_KEY"))
openai_model = ChatOpenAI(model="gpt-5-nano", api_key= openai_api_key, reasoning_effort="low").bind_tools(tools, tool_choice='auto')

In [66]:
openai_model.invoke(
    input=[HumanMessage("How Are you")]
)

AIMessage(content="I'm doing well, thanks for asking! How can I help you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 186, 'total_tokens': 210, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EEKcoBRM9vedEytQCV7HMp4pjygL6', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0168a-ac6a-7610-a3a1-eaa25b29662b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 186, 'output_tokens': 24, 'total_tokens': 210, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [67]:
openai_model.invoke(
    input=[HumanMessage("What is the database status?")]
)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 189, 'total_tokens': 278, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EEKcrYU1pldvXCCVmJd5ZOnaH1Wlp', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0168a-b916-73b2-8926-44e2854cf11d-0', tool_calls=[{'name': 'get_database_status', 'args': {'cluster_name': 'default'}, 'id': 'call_asWaIoC801RruLhvcyDcez5t', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 189, 'output_tokens': 89, 'total_tokens': 278, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio

In [68]:
from langchain_core.messages import BaseMessage

def run_agent_loop(conversation: List[BaseMessage]):
    MAX_ITERATIONS = 100
    finished_successfully = False
    for i in range(MAX_ITERATIONS):
        reply = openai_model.invoke(conversation)
        conversation.append(reply)

        if not reply.tool_calls:
            finished_successfully = True
            break

        for tool_call in reply.tool_calls:
            print("Tool call executing")
            print(tool_call)

            tool_call_id = tool_call["id"]
            tool_call_name = tool_call["name"]
            tool_call_args = tool_call["args"]

            result = tools_registry[tool_call_name].invoke(tool_call_args)
            conversation.append(ToolMessage(str(result), tool_call_id = tool_call_id))

    if not finished_successfully:
        raise RuntimeError(f"Could not finish the interaction withing {MAX_ITERATIONS} iterations.")

In [69]:
query_database_status = [
    #SystemMessage("You are a helpful solution monitoring assistant. When using the `get_database_status` tool, if the cluster name is unknown, use \"default\"."),
    HumanMessage("What is the current status of my database cluster?")
]

run_agent_loop(query_database_status)

Tool call executing
{'name': 'get_database_status', 'args': {'cluster_name': 'default'}, 'id': 'call_t18WMUKPoX56IxAo7Tubibl1', 'type': 'tool_call'}


In [47]:
query_database_status

[HumanMessage(content='What is the current status of my database cluster?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 153, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EEKKtoXaIxD97P13cnRNVz7dXKcsk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a01679-b757-7b51-9e39-cb4e1e114de0-0', tool_calls=[{'name': 'get_database_status', 'args': {'cluster_name': 'default'}, 'id': 'call_Avz21tnTGHmLTIP8P9CWvKNo', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'outpu

In [52]:
print_conversation(query_database_status)

================================ Human Message =================================

What is the current status of my database cluster?
================================== Ai Message ==================================
Tool Calls:
  get_database_status (call_oW5gNfkQc3VSVFHlmhjcecjV)
 Call ID: call_oW5gNfkQc3VSVFHlmhjcecjV
  Args:
    cluster_name: default
================================= Tool Message =================================

healthy
================================== Ai Message ==================================

The database cluster "default" is healthy. 

If you’d like more details (uptime, replication status, resource usage, error logs), I can fetch them.
